# INHALEX — Predicción mensual de demanda por producto

Esta libreta reproduce la propuesta 2 mediante una regresión Ridge global. Una fila histórica representa un producto durante un mes objetivo y la variable Y es `Y_unidades_solicitadas_mes`.

La evaluación es cronológica: los últimos tres meses se reservan para validación y nunca se mezclan aleatoriamente con meses anteriores.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

REPO_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'ml').is_dir() and (path / 'Server').is_dir()
)
EXPORT_DIR = REPO_ROOT / 'ml' / 'exports'
ARTIFACT_DIR = REPO_ROOT / 'ml' / 'artifacts'

## 1. Entrenamiento y pronóstico reproducible

El script reconstruye las 16 filas de julio de 2026 usando únicamente información disponible hasta el 30 de junio. Después evalúa Ridge frente al promedio móvil de tres meses, entrena con todo el histórico y exporta el pronóstico.

In [ ]:
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'ml' / 'src' / 'train_monthly_demand.py')],
    cwd=REPO_ROOT,
    check=True,
)

## 2. Filas futuras sin variable Y

`fecha_corte` indica hasta cuándo se permitió formar X. En producción, Y todavía no existe para julio; será conocida al finalizar ese mes.

In [ ]:
future = pd.read_csv(
    EXPORT_DIR / 'dataset_demanda_inferencia.csv',
    encoding='utf-8-sig',
)
future[['producto', 'mes_objetivo', 'demanda_lag_1m', 'demanda_lag_2m', 'demanda_lag_3m', 'promedio_demanda_3m']].head(8)

## 3. Métricas temporales y salida del modelo

MAE expresa cuántas unidades se desvía el modelo en promedio. RMSE penaliza con mayor fuerza los errores grandes. El baseline es el promedio de los tres meses anteriores.

In [ ]:
artifact = json.loads(
    (ARTIFACT_DIR / 'monthly-demand-forecast.v1.json').read_text(encoding='utf-8')
)
metrics = pd.Series(artifact['model']['metrics'], name='valor')
display(metrics.to_frame())

forecast = pd.DataFrame([
    {
        'producto': item['productName'],
        'mes_objetivo': artifact['targetMonth'],
        'prediccion': item['prediction']['units'],
        'limite_inferior': item['prediction']['lower'],
        'limite_superior': item['prediction']['upper'],
    }
    for item in artifact['items']
]).sort_values('prediccion', ascending=False)
display(forecast)

## 4. Puerta de calidad

Se valida que exista una salida por producto, que las predicciones sean no negativas, que los intervalos estén ordenados y que Ridge mejore el baseline temporal.

In [ ]:
assert len(future) == 16
assert future['product_id'].is_unique
assert artifact['targetMonth'] == future['mes_objetivo'].iloc[0]
assert artifact['model']['metrics']['mae'] <= artifact['model']['metrics']['baselineMae']
assert all(
    0 <= item['prediction']['lower'] <= item['prediction']['units'] <= item['prediction']['upper']
    for item in artifact['items']
)
print('APROBADO: regresión mensual y pronósticos listos para integración.')